# Fit 3 — TNG300-1 Halo Fractal Dimension

This notebook measures $N(\varepsilon) \propto \varepsilon^{-D}$ for dark matter halos from IllustrisTNG-300-1.

**Goal:** Get D in the range 2.5–2.7 to test the open-source prediction.

Run time: ~5 minutes once data is downloaded.

## Step 0 — Install (run once)
```bash
pip install illustris_python h5py numpy scipy matplotlib
```

## Step 1 — Get the data

1. Register free at https://www.tng-project.org
2. Download TNG300-1, snapshot 99 (z=0), file `groups_099.hdf5` (~3 GB)
3. Or use the API: set your API key below

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.stats import linregress

# --- EDIT THIS PATH ---
hdf5_path = 'groups_099.hdf5'  # put your local path here

# Load halo positions and masses
with h5py.File(hdf5_path, 'r') as f:
    pos = f['Group']['GroupPos'][:]  # ckpc/h
    mass = f['Group']['GroupMass'][:] * 1e10  # Msun/h

# Select massive halos > 1e13 Msun/h (clusters)
mask = mass > 1e13
pos = pos[mask] / 1000.0  # convert to Mpc/h
print(f'Using {len(pos)} halos')

# Box size for TNG300 is 205 Mpc/h
Lbox = 205.0

## Step 2 — Box-counting function

In [ ]:
def box_count(positions, Lbox, epsilons):
    """Count occupied boxes at each scale epsilon"""
    Ns = []
    for eps in epsilons:
        ngrid = int(np.ceil(Lbox / eps))
        # assign each halo to a grid cell
        ix = np.floor(positions[:,0] / eps).astype(int)
        iy = np.floor(positions[:,1] / eps).astype(int)
        iz = np.floor(positions[:,2] / eps).astype(int)
        # count unique occupied cells
        keys = ix + ngrid*(iy + ngrid*iz)
        N = len(np.unique(keys))
        Ns.append(N)
    return np.array(Ns)

# Scales to test (Mpc/h) - matches Fit 3 spec
epsilons = np.array([2, 5, 10, 20, 50])
Ns = box_count(pos, Lbox, epsilons)

## Step 3 — Fit D

In [ ]:
log_eps = np.log10(epsilons)
log_N = np.log10(Ns)

slope, intercept, r, p, se = linregress(log_eps, log_N)
D = -slope

print(f'Fit: log N = {intercept:.2f} + ({slope:.2f}) log ε')
print(f'Fractal dimension D = {D:.3f} ± {se:.3f}')
print(f'R² = {r**2:.3f}')

## Step 4 — Plot for Facebook

In [ ]:
plt.figure(figsize=(5,4))
plt.loglog(epsilons, Ns, 'o-', label='TNG300 halos')
plt.loglog(epsilons, 10**intercept * epsilons**slope, '--', label=f'Fit D={D:.2f}')
plt.xlabel('ε [Mpc/h]')
plt.ylabel('N(ε)')
plt.title('Fractal Scaling — TNG300-1')
plt.grid(True, which='both', ls=':')
plt.legend()
plt.tight_layout()
plt.savefig('tng_fit.png', dpi=150)
plt.show()

## Step 5 — Post your result

Copy this into the Facebook thread:

> **FIT 3 — TNG300-1 halos** | D = [your value] ± [error] | ε = 2–50 Mpc/h | R² = [value]

Upload `tng_fit.png` as your proof.

---
**No TNG data yet?** Run the cells below to test the code on a synthetic fractal (D=2.6) — then replace with real halos later.

In [ ]:
# Synthetic test (optional)
np.random.seed(42)
N_synth = 20000
# generate points with fractal-like clustering (simple power-law sampling)
r = np.random.power(2.6, N_synth) * Lbox/2
theta = np.random.uniform(0, 2*np.pi, N_synth)
phi = np.arccos(np.random.uniform(-1,1, N_synth))
x = Lbox/2 + r*np.sin(phi)*np.cos(theta)
y = Lbox/2 + r*np.sin(phi)*np.sin(theta)
z = Lbox/2 + r*np.cos(phi)
pos_test = np.vstack([x,y,z]).T

Ns_test = box_count(pos_test, Lbox, epsilons)
slope_t, _, r_t, _, se_t = linregress(np.log10(epsilons), np.log10(Ns_test))
print(f'Synthetic D = {-slope_t:.3f} (should be ~2.6)')